# Sandbox vs Production comparison

Compares dbt-built Sandbox objects against their production Silver/Gold counterparts for the
`Claim_Fact` -> `Claim_Aggr` migration line. No primary key is defined on either object, so this
notebook is limited to checks that do not require row-level alignment:

- column names match (`compare_columns` from `Lib_Westfund`)
- row counts match
- pandas dtypes match per column
- null percentage per column
- sum of numeric columns
- unique value sets of categorical columns

In [ ]:
import sys
import os
import pandas as pd
import pyodbc

sys.path.append(os.path.abspath(os.path.join('..', '..')))

from Lib_Westfund import Logger, compare_columns

## Config

Database / schema / table names as variables so this notebook can be reused for other lines.

In [ ]:
SERVER = 'prdsql05.westfund.com.au'

# (sandbox_database, sandbox_schema, sandbox_table, prod_database, prod_schema, prod_table)
COMPARISONS = [
    # ('SANDBOX', 'stg', 'person', 'BRONZE', 'dbo', 'person'),
    ('SANDBOX', 'mart', 'Claim_Aggr', 'GOLD', 'dbo', 'Claim_Aggr'),
]

In [ ]:
def get_connection(database):
    return pyodbc.connect(
        'DRIVER={ODBC Driver 17 for SQL Server};'
        f'SERVER={SERVER};'
        f'DATABASE={database};'
        'Trusted_Connection=yes;'
    )


def read_table(database, schema, table):
    conn = get_connection(database)
    query = f'SELECT * FROM [{database}].[{schema}].[{table}]'
    df = pd.read_sql(query, conn)
    conn.close()
    return df

## Comparison functions

Each function logs via `Logger` (matching `Lib_Westfund` style) and also returns a `DataFrame`
with the detailed, row-by-row comparison result, so results can be displayed as readable tables
instead of just log lines. `compare_columns` is reused from `Lib_Westfund` for the log output;
the column-level detail table is built separately here.

In [ ]:
def compare_columns_detail(logger, old_df, new_df):
    compare_columns(logger, old_df, new_df)

    old_cols = set(old_df.columns)
    new_cols = set(new_df.columns)
    all_cols = sorted(old_cols | new_cols)

    rows = []
    for col in all_cols:
        if col in old_cols and col in new_cols:
            status = 'both'
        elif col in old_cols:
            status = 'prod only'
        else:
            status = 'sandbox only'
        rows.append({'column': col, 'status': status})

    return pd.DataFrame(rows)


def compare_row_counts_detail(logger, old_df, new_df):
    logger.debug('Comparing row counts', True)
    old_n, new_n = len(old_df), len(new_df)
    diff = new_n - old_n
    if diff == 0:
        logger.debug(f'Row counts match: {old_n:,}')
    else:
        logger.error(f'Row counts differ: prod={old_n:,} sandbox={new_n:,} (diff={diff:,})')

    return pd.DataFrame([{
        'prod_rows': old_n,
        'sandbox_rows': new_n,
        'diff': diff,
        'match': diff == 0,
    }])


def compare_dtypes_detail(logger, old_df, new_df):
    logger.debug('Comparing dtypes', True)
    common_cols = sorted(set(old_df.columns) & set(new_df.columns))

    rows = []
    n_issues = 0
    for col in common_cols:
        old_dtype = old_df[col].dtype
        new_dtype = new_df[col].dtype
        match = old_dtype == new_dtype
        if not match:
            logger.warning(f'Column {col} dtype differs: prod={old_dtype} sandbox={new_dtype}')
            n_issues += 1
        rows.append({'column': col, 'prod_dtype': str(old_dtype), 'sandbox_dtype': str(new_dtype), 'match': match})
    if n_issues == 0:
        logger.debug('Dtypes match exactly')

    return pd.DataFrame(rows)


def compare_null_rates_detail(logger, old_df, new_df):
    logger.debug('Comparing null percentages', True)
    common_cols = sorted(set(old_df.columns) & set(new_df.columns))

    rows = []
    for col in common_cols:
        old_pct = old_df[col].isna().mean() * 100
        new_pct = new_df[col].isna().mean() * 100
        diff = new_pct - old_pct
        match = abs(diff) <= 0.01
        if not match:
            logger.warning(f'Column {col} null%: prod={old_pct:.2f}% sandbox={new_pct:.2f}%')
        rows.append({
            'column': col,
            'prod_null_pct': round(old_pct, 2),
            'sandbox_null_pct': round(new_pct, 2),
            'diff_pct': round(diff, 2),
            'match': match,
        })

    return pd.DataFrame(rows)


def is_key_col(col_name):
    return col_name.lower().endswith('_id') or col_name.lower() == 'id'


def get_numeric_cols(df):
    return [c for c in df.columns if not is_key_col(c) and df[c].dtype in ('int64', 'float64')]


def get_categorical_cols(df, max_unique=50):
    cols = []
    for c in df.columns:
        if is_key_col(c) or df[c].dtype != 'object':
            continue
        if df[c].nunique(dropna=True) <= max_unique:
            cols.append(c)
    return cols


def compare_numeric_sums_detail(logger, old_df, new_df):
    logger.debug('Comparing numeric column sums', True)
    common_numeric = sorted(set(get_numeric_cols(old_df)) & set(get_numeric_cols(new_df)))

    rows = []
    for col in common_numeric:
        old_sum = old_df[col].sum()
        new_sum = new_df[col].sum()
        pct_diff = 0.0 if old_sum == 0 else abs(new_sum - old_sum) / abs(old_sum) * 100
        match = pct_diff <= 0.01
        if match:
            logger.debug(f'Column {col} sum matches: prod={old_sum:,.2f} sandbox={new_sum:,.2f}')
        else:
            logger.error(f'Column {col} sum differs: prod={old_sum:,.2f} sandbox={new_sum:,.2f} (diff={pct_diff:.4f}%)')
        rows.append({
            'column': col,
            'prod_sum': round(old_sum, 2),
            'sandbox_sum': round(new_sum, 2),
            'diff_pct': round(pct_diff, 4),
            'match': match,
        })

    return pd.DataFrame(rows)


def compare_categorical_values_detail(logger, old_df, new_df):
    logger.debug('Comparing categorical value sets', True)
    common_categorical = sorted(set(get_categorical_cols(old_df)) & set(get_categorical_cols(new_df)))

    rows = []
    for col in common_categorical:
        old_vals = set(old_df[col].dropna().unique())
        new_vals = set(new_df[col].dropna().unique())
        old_only = old_vals - new_vals
        new_only = new_vals - old_vals
        match = not old_only and not new_only
        if old_only:
            logger.warning(f'Column {col} values in prod not in sandbox: {old_only}')
        if new_only:
            logger.warning(f'Column {col} values in sandbox not in prod: {new_only}')
        rows.append({
            'column': col,
            'prod_values': sorted(old_vals, key=str),
            'sandbox_values': sorted(new_vals, key=str),
            'prod_only': sorted(old_only, key=str),
            'sandbox_only': sorted(new_only, key=str),
            'match': match,
        })

    return pd.DataFrame(rows)

## Run comparisons

In [ ]:
logger = Logger()
results = {}

for sbx_db, sbx_schema, sbx_table, prod_db, prod_schema, prod_table in COMPARISONS:
    label = f'{sbx_schema}.{sbx_table}'
    logger.debug(f'===== {prod_db}.{prod_schema}.{prod_table}  vs  {sbx_db}.{sbx_schema}.{sbx_table} =====', True)

    prod_df = read_table(prod_db, prod_schema, prod_table)
    sbx_df = read_table(sbx_db, sbx_schema, sbx_table)

    results[label] = {
        'columns': compare_columns_detail(logger, prod_df, sbx_df),
        'row_count': compare_row_counts_detail(logger, prod_df, sbx_df),
        'dtypes': compare_dtypes_detail(logger, prod_df, sbx_df),
        'null_rates': compare_null_rates_detail(logger, prod_df, sbx_df),
        'numeric_sums': compare_numeric_sums_detail(logger, prod_df, sbx_df),
        'categorical_values': compare_categorical_values_detail(logger, prod_df, sbx_df),
    }

## View results

`results` is a dict keyed by `"schema.table"` (e.g. `itm.Claim_Fact`), each holding the 6 detail
DataFrames: `columns`, `row_count`, `dtypes`, `null_rates`, `numeric_sums`, `categorical_values`.
Pick a table below to inspect.

In [ ]:
for table, detail in results.items():
    print(f'\n========== {table} ==========')

    print('columns:')
    display(detail['columns'])

    print('row_count:')
    display(detail['row_count'])

    print('dtypes:')
    display(detail['dtypes'])

    print('null_rates:')
    display(detail['null_rates'])

    print('numeric_sums:')
    display(detail['numeric_sums'])

    print('categorical_values:')
    display(detail['categorical_values'])

## note

In [ ]:
# .venv\Scripts\dbt.exe build --select staging+

In [ ]:
# USE SANDBOX;
# GO

# DECLARE @sql NVARCHAR(MAX) = N'';

# SELECT @sql = @sql +
#     'DROP ' +
#     CASE o.type WHEN 'V' THEN 'VIEW ' ELSE 'TABLE ' END +
#     QUOTENAME(s.name) + '.' + QUOTENAME(o.name) + ';' + CHAR(13)
# FROM sys.objects o
# JOIN sys.schemas s ON o.schema_id = s.schema_id
# WHERE s.name IN ('stg', 'itm', 'mart')
#   AND o.type IN ('V', 'U');

# PRINT @sql;
# EXEC sp_executesql @sql;
